In [1]:
import pandas as pd
from glob import glob

from _analysis import load_jsons

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth


/home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model_path,

In [2]:
usr_path = "/home/weissl"
mimicry_i_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_c*/*.csv")]
mimicry_c_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_b*/*.csv")]

hynea_i_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_1*/*.csv")]
hynea_c_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_b*/*.csv")]
hynea_y_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_y*/*.csv")]

In [3]:
mimicry_i = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_c*/*.json"))
mimicry_c = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_b*/*.json"))

hynea_i = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_1*/*.json"))
hynea_c = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_b*/*.json"))
hynea_y = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_y*/*.json"))

mrm_path = f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_"
mrm_i = load_jsons(glob(mrm_path + "/*/*.json"))
mrm_i["runtime"] = mrm_i["runtime_seconds"]

mrm_c = load_jsons(glob(f"{usr_path}/PycharmProjects/genai_tigs/sd_weights/celebahq_generatorlow/*.json"))
mrm_c["runtime"] = mrm_c["runtime_seconds"]

mrm_y = load_jsons(glob(f"{usr_path}/PycharmProjects/genai_tigs/yolo_sd/test_/*/*.json"))
mrm_y["runtime"] = mrm_y["runtime_seconds"]

In [4]:
def get_safe_times(df, thresh: int = 30_000):
    """Sometimes time is glitched when logging -> remove those glitched readings."""
    return df["runtime"][df["runtime"] < thresh]

"""Get stats for mimicry"""
mimicry_i_budget = mimicry_i["budget_used"] + mimicry_i["w0_trials"] + mimicry_i["wn_trials"]
mimicry_i_runtime = get_safe_times(mimicry_i)

mimicry_c_budget = mimicry_c["budget_used"] + mimicry_c["w0_trials"] + mimicry_c["wn_trials"]
mimicry_c_runtime = get_safe_times(mimicry_c)

"""Get stats for hynea"""
hynea_i_budget = hynea_i["budget_used"]
hynea_i_runtime = get_safe_times(hynea_i)

hynea_c_budget = hynea_c["budget_used"]
hynea_c_runtime = get_safe_times(hynea_c)

hynea_y_budget = hynea_y["budget_used"]
hynea_y_runtime = get_safe_times(hynea_y)

"""Get stats for GIFTbench"""
mrm_i_runtime = get_safe_times(mrm_i)
mrm_i_budget = mrm_i["budget"]

mrm_c_runtime = get_safe_times(mrm_c)
mrm_c_budget = mrm_c["budget"]

mrm_y_runtime = get_safe_times(mrm_y)
mrm_y_budget = mrm_y["budget"]

"""
Get extrapolate stats for mimicry with diffusion

>%%timeit
>with torch.no_grad():
>   manipulator.get_diff_steps([1]*100)

>%%timeit
>with torch.no_grad():
>    manipulator.get_diff_steps([1]*100)
"""
extrapolate_i = (mimicry_i_budget * 45.9) / 100
extrapolate_c = (mimicry_c_budget * 47.2) / 100

In [5]:
tools = ["HyNeA", "mimicry", "GIFTbench"]
experiments = ["ImageNet", "CelebA", "Guericke"]

In [6]:
print("Budget Used:\n")
print(f"Hynea ImageNet: {hynea_i_budget.mean():.2f}, {hynea_i_budget.std():.2f}")
print(f"Hynea CelebA: {hynea_c_budget.mean():.2f}, {hynea_c_budget.std():.2f}")
print(f"Hynea Guericke: {hynea_y_budget.mean():.2f}, {hynea_y_budget.std():.2f}\n")

print(f"Mimicry ImageNet: {mimicry_i_budget.mean():.2f}, {mimicry_i_budget.std():.2f}")
print(f"Mimicry CelebA: {mimicry_c_budget.mean():.2f}, {mimicry_c_budget.std():.2f}\n")

print(f"GIFTbench Imagenet: {mrm_i_budget.mean():.2f}, {mrm_i_budget.std():.2f}")
print(f"GIFTbench CelebA: {mrm_c_budget.mean():.2f}, {mrm_c_budget.std():.2f}\n")
print(f"GIFTbench Guericke: {mrm_y_budget.mean():.2f}, {mrm_y_budget.std():.2f}")

Budget Used:

Hynea ImageNet: 25.29, 26.72
Hynea CelebA: 30.30, 31.43
Hynea Guericke: 5.71, 5.69

Mimicry ImageNet: 2498.16, 390.57
Mimicry CelebA: 2672.25, 38.12

GIFTbench Imagenet: 699.63, 748.59
GIFTbench CelebA: 1796.00, 1088.11

GIFTbench Guericke: 232.12, 245.72


In [7]:
print("Runtime: \n")
print(f"Hynea ImageNet: {hynea_i_runtime.mean():.2f}, {hynea_i_runtime.std():.2f}")
print(f"Hynea CelebA: {hynea_c_runtime.mean():.2f}, {hynea_c_runtime.std():.2f}")
print(f"Hynea Guericke: {hynea_y_runtime.mean():.2f}, {hynea_y_runtime.std():.2f}\n")

print(f"Mimicry ImageNet: {mimicry_i_runtime.mean():.2f}, {mimicry_i_runtime.std():.2f}")
print(f"Mimicry CelebA: {mimicry_c_runtime.mean():.2f}, {mimicry_c_runtime.std():.2f}\n")

print(f"Extrapolate Imagenet: {extrapolate_i.mean():1.0f}")
print(f"Extrapolate CelebA: {extrapolate_c.mean():1.0f}\n")

print(f"GIFTbench Imagenet: {mrm_i_runtime.mean():.2f}, {mrm_i_runtime.std():.2f}")
print(f"GIFTbench CelebA: {mrm_c_runtime.mean():.2f}, {mrm_c_runtime.std():.2f}")
print(f"GIFTbench Guericke: {mrm_y_runtime.mean():.2f}, {mrm_y_runtime.std():.2f}")

Runtime: 

Hynea ImageNet: 94.41, 99.26
Hynea CelebA: 220.89, 228.82
Hynea Guericke: 113.03, 111.70

Mimicry ImageNet: 108.53, 16.93
Mimicry CelebA: 51.79, 3.53

Extrapolate Imagenet: 1147
Extrapolate CelebA: 1261

GIFTbench Imagenet: 205.74, 198.99
GIFTbench CelebA: 1217.24, 735.71
GIFTbench Guericke: 174.58, 165.02
